# Implementing a Hook

> **Erratum in the course:** the *video* shows an older version that also covered Grep — the **current text is Read-only**, which is what we follow here.

We build a **PreToolUse** hook that blocks the **Read** tool from opening `.env`, protecting environment variables during a session.


## Scope: Read only

This hook covers the **Read** tool only. Blocking **Grep** or **Bash** from reaching the same file needs separate handling (each tool sends a different input shape). For comprehensive file protection, **combine a hook with a `permissions.deny` rule** like `"Read(**/.env)"`, which applies uniformly across tools.

## 1. Configure the hook

In `.claude/settings.local.json`, add a PreToolUse hook matching the `Read` tool:

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Read",
        "hooks": [
          { "type": "command", "command": "node $PWD/hooks/read_hook.js" }
        ]
      }
    ]
  }
}
```

## 2. Write the hook script

`hooks/read_hook.js`:

```javascript
process.stdin.setEncoding("utf8");
let input = "";
process.stdin.on("data", (d) => (input += d));
process.stdin.on("end", () => {
  const toolArgs = JSON.parse(input);
  const readPath = toolArgs.tool_input?.file_path || "";
  if (readPath.includes(".env")) {
    console.error("You cannot read the .env file");
    process.exit(2);
  }
  process.exit(0);
});
```

It reads the tool call JSON from **stdin**, checks `tool_input.file_path`, and **exits `2`** to block (whatever goes to **stderr** becomes the message Claude sees). Exit `0` = allow.

## 3. Test it

In a Claude Code session (inside the `queries` project), ask Claude to read the `.env` file. You should see:

```
You cannot read the .env file
```

That's the hook blocking the Read. Ask it to read a *different* file → works normally.

## Why Read only?

Each tool sends a different `tool_input` shape:

| Tool | `tool_input` |
|------|--------------|
| **Read** | `{ "file_path": "..." }` |
| **Grep** | `{ "pattern": "...", "path": "..." }` (path = search dir, not a file) |
| **Bash** | `{ "command": "..." }` |

A check on `file_path` catches **Read** but not a project-wide `grep` for `API_KEY` or a `cat .env` in **Bash**. To cover those, write **separate matchers per tool** (inspecting each one's fields) — or use **`permissions.deny`** rules, which apply uniformly.

## In this repo

- **Answer key:** [`../03-hooks-and-the-sdk/queries-complete/hooks/read_hook.js`](./queries-complete/hooks/read_hook.js) is set to the **current** lesson script (the zip shipped the older `for await` variant), and `queries-complete/.claude/settings.local.json` holds the Read-hook config above.
- **Your exercise:** implement the `.env` check in `queries/hooks/read_hook.js` (it's `skip-worktree`, so your edit stays local) and create `queries/.claude/settings.local.json`.
- **Node required** to run the hook (it's a Node script).

Next: **Gotchas around hooks**.